# Feature Engineering Study - NLP (Classificacao)Estudo comparativo de representacoes textuais para classificacao de sentimentono dataset **Twitter Entity Sentiment Analysis** (73.768 treino, 999 validacao,4 classes). Cada representacao e avaliada com um modelo fixo (**LinearSVC**),isolando o efeito da feature engineering.Pergunta central: *qual representacao textual e mais discriminativa?*

In [1]:
import numpy as npimport pandas as pdimport reimport timeimport osimport warningswarnings.filterwarnings('ignore')from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, HashingVectorizerfrom sklearn.preprocessing import LabelEncoderfrom sklearn.svm import LinearSVCfrom sklearn.metrics import accuracy_score, f1_scoreimport kagglehubSEED = 42np.random.seed(SEED)print('Imports OK')

Imports OK


## 1. Carregamento e Pre-processamento

In [2]:
path = kagglehub.dataset_download('jp797498e/twitter-entity-sentiment-analysis')train_df = pd.read_csv(os.path.join(path, 'twitter_training.csv'))val_df = pd.read_csv(os.path.join(path, 'twitter_validation.csv'))COL_NAMES = ['id', 'entity', 'sentiment', 'text']if train_df.shape[1] == 4:    train_df.columns = COL_NAMES    val_df.columns = COL_NAMESdef clean_tweet(text):    if not isinstance(text, str): return ''    text = text.lower()    text = re.sub(r'http\S+|www\S+', '', text)    text = re.sub(r'@\w+', '', text)    text = re.sub(r'#(\w+)', r'\1', text)    text = re.sub(r'[^a-z0-9\s!?.,\'\-\"]', '', text)    text = re.sub(r'\s+', ' ', text).strip()    return textfor df in (train_df, val_df):    df['clean_text'] = df['text'].apply(clean_tweet)valid_sentiments = ['Positive', 'Negative', 'Neutral', 'Irrelevant']train_df = train_df[train_df['clean_text'].str.len() > 0 & train_df['sentiment'].isin(valid_sentiments)].copy()val_df = val_df[val_df['clean_text'].str.len() > 0 & val_df['sentiment'].isin(valid_sentiments)].copy()le = LabelEncoder()le.fit(valid_sentiments)y_train = le.transform(train_df['sentiment'])y_val = le.transform(val_df['sentiment'])X_train_text = train_df['clean_text'].valuesX_val_text = val_df['clean_text'].valuesprint(f'Train: {len(X_train_text)}, Val: {len(X_val_text)}')

Train: 73768, Val: 999


## 2. Representacoes Avaliadas| # | Representacao | Descricao ||---|---------------|-----------|| 1 | BoW (CountVectorizer) | Contagem bruta de termos || 2 | TF-IDF (1-gram) | TF-IDF unigramas || 3 | TF-IDF (1-2 gram) | TF-IDF uni + bigramas || 4 | TF-IDF (1-3 gram) | TF-IDF uni + bi + trigramas || 5 | TF-IDF char (2-5) | TF-IDF char n-gramas 2-5 || 6 | TF-IDF char (3-5) | TF-IDF char n-gramas 3-5 || 7 | Hashing trick (2^18) | HashingVectorizer 262k dims || 8 | Combined word+char | TF-IDF word (50k) + char (20k) || 9 | Sentence-BERT | all-MiniLM-L6-v2 frozen (384-d) |Modelo fixo: **LinearSVC (C=1.0)** para todas as representacoes.

## 3. Resultados

In [3]:
# Resultados da execucao completa (script _run_fe_nlp.py)

Representation                 Acc      F1-w     Feat       Time
--------------------------------------------------------------------
1. BoW (CountVectorizer)       0.9550   0.9550   26351      125.86
2. TF-IDF (1-gram)             0.9419   0.9419   26351      9.01
3. TF-IDF (1-2 gram)           0.9770   0.9770   70000      18.84
4. TF-IDF (1-3 gram)           0.9780   0.9780   70000      15.40
5. TF-IDF char (2-5)           0.9459   0.9460   70000      69.71
6. TF-IDF char (3-5)           0.9439   0.9440   70000      55.42
7. Hashing trick (2^18)        0.9860   0.9860   262144     8.61
8. Combined word+char          0.9820   0.9820   70000      60.67
9. Sentence-BERT               0.6246   0.6129   384        150.72


## 4. Tabela Resumo (ordenado por acuracia)| Representacao | Acuracia | F1-w | Features | Tempo (s) ||---------------|----------|------|----------|-----------|| **Hashing trick (2^18)** | **0,9860** | **0,9860** | 262.144 | 8,61 || Combined word+char | 0,9820 | 0,9820 | 70.000 | 60,67 || TF-IDF (1-3 gram) | 0,9780 | 0,9780 | 70.000 | 15,40 || TF-IDF (1-2 gram) | 0,9770 | 0,9770 | 70.000 | 18,84 || BoW (CountVectorizer) | 0,9550 | 0,9550 | 26.351 | 125,86 || TF-IDF char (2-5) | 0,9459 | 0,9460 | 70.000 | 69,71 || TF-IDF char (3-5) | 0,9439 | 0,9440 | 70.000 | 55,42 || TF-IDF (1-gram) | 0,9419 | 0,9419 | 26.351 | 9,01 || Sentence-BERT | 0,6246 | 0,6129 | 384 | 150,72 |

## 5. Analise### 1. Hashing Trick - O Vencedor Surpreendente (0,9860 em 8,6s)O HashingVectorizer com 2^18 = 262.144 dimensoes superou todas as outrasrepresentacoes. As razoes do sucesso:- **mais dimensoes que TF-IDF** (262k vs 70k), reduzindo colisoes de hash- **sem precisao de IDF**: a ausencia do calculo de IDF acelera o fit (8,61s vs18,84s do TF-IDF), e o SVM linear compensa via margem de separacao- **sem vocabulario**: nao precisa armazenar o dicionario, ideal para streaming### 2. TF-IDF: N-gramas Importam (+3,5 pp)| Representacao | Acuracia | Ganho vs 1-gram ||---------------|----------|-----------------|| TF-IDF (1-gram) | 0,9419 | - || TF-IDF (1-2 gram) | 0,9770 | +3,5 pp || TF-IDF (1-3 gram) | 0,9780 | +3,6 pp |Bigramas capturam expressoes como "nao gostei" e "muito bom" que unigramasperdem. Trigramas dao ganho marginal (+0,1 pp) mas aumentam o tempo.### 3. Char n-gramas: Inferiores a Word n-gramasChar n-gramas (0,9459 para 2-5) ficam abaixo de word n-gramas (0,9770 para 1-2).Char n-gramas sao uteis para textos com typos e Regionalismos, mas neste datasetlimpo (pre-processado) eles adicionam ruido sem beneficio. No entanto, acombinacao **word + char** (0,9820) supera ambos isoladamente, mostrando quechar n-gramas trazem informacao complementar.### 4. BoW vs TF-IDF: IDF Atrapalha Aqui?Surpreendentemente, BoW (CountVectorizer, 0,9550) supera TF-IDF 1-gram (0,9419)em +1,3 pp. Isso parece contra-intuitivo, mas pode ser explicado pela distribuicao:como o dataset contem muitas repeticoes (tweets sobre as mesmas marcas), afrequencia bruta de termos relevantes ("kill", "murder", "love") e maisdiscriminativa que a ponderacao IDF. Contudo, BoW e **14x mais lento** para treinar(125,86s vs 9,01s), pois as contagens densas exigem mais iteracoes do SVM.### 5. Sentence-BERT: Frozen Continua Inadequado (0,6246)Confirma o achado do notebook NLP-twitter-methods-comparasion: o embeddinggenerico de 384-d do all-MiniLM-L6-v2 nao captura polaridade afetiva.Importante: aqui o modelo e LinearSVC (nao LinearSVC com calibracao - o resultadoe ligeiramente pior que o 0,6036 do notebook anterior por usar C=1.0 fixo).### Conclusoes1. **Hashing trick e o melhor custo-beneficio**: 0,9860 em 8,6s, sem necessidade   de construir vocabulario. Ideal para pipelines de producao.2. **TF-IDF com n-gramas (1-2) e o padrao recomendado**: 0,9770 em 18,8s,   equilibrio otimo entre acuracia e interpretabilidade ( IDF e analysable).3. **Combinar word + char n-gramas traz ganho real** (+0,5 pp sobre TF-IDF word   sozinho), mostrando que captamos padroes ortograficos complementares.4. **Char n-gramas isoladamente sao inferiores a word n-gramas**, mas uteis em   conjunto ou para textos ruidosos (typos, grias).5. **Sentence-BERT frozen nao serve para classificacao de sentimento** - confirmed   pelo segundo experimento independente.6. **Recomendacao final**: Use Hashing trick para maxima acuracia em minimo tempo.   Use TF-IDF (1-2 gram) quando precisar de interpretabilidade. Combine word + char   quando quiser extrair o maximo de performance classica.